# Prepare Task 1 with cleaned dataset1

Run **after** `notebooks/00_eda_and_preprocessing.ipynb`. This notebook reads its `preprocessed_datasets/train_manifest.csv`, freezes the existing supplied-only Task 1 split, and adds the 1,158 retained external crops **to training only**. It does not change the EDA manifest, source images, training notebooks, or checkpoints.

Run all cells from the repository root or this Task1 folder using the project environment. Outputs go into a versioned directory under `preprocessed_datasets/task1_dataset1/`. Dataset2 is excluded. Source/licence provenance remains to be established; a clean visual screen is not independent label authentication.


In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import pandas as pd
from PIL import Image
from IPython.display import display

REPO_ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "src/preprocessing.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Run this notebook from within the assignment repository.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.preprocessing import load_manifest, make_split, standardize_image, IMAGE_TARGET_SIZE

TARGET = "articleType"
SPLIT_SEED = 42
VALIDATION_SHARE = 0.2
BASE_CSV = REPO_ROOT / "preprocessed_datasets/train_manifest.csv"
EXTERNAL_ROOT = REPO_ROOT / "notebooks/Task1/dataset1"
EXTERNAL_CSV = EXTERNAL_ROOT / "external_cosmetics.csv"
OUTPUT_ROOT = REPO_ROOT / "preprocessed_datasets/task1_dataset1"

def require(condition, message):
    if not condition:
        raise ValueError(message)

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

for path in (BASE_CSV, EXTERNAL_CSV, EXTERNAL_ROOT / "audit/technical_summary.json",
             EXTERNAL_ROOT / "audit/image_audit.csv", EXTERNAL_ROOT / "audit/removal_manifest.json"):
    if not path.is_file():
        raise FileNotFoundError(f"Missing input: {path}. Run EDA first and retain dataset1 audit records.")
print("Repository:", REPO_ROOT)


## 1. Validate the cleaned external snapshot

Check the CSV and every retained image against the post-cleanup audit, including removed-ID exclusion, complete image/label pairing, and decoding through the shared RGB/60x80 transform. The earlier audit found no exact or dHash-radius-2 matches to supplied train/test or dataset2; this notebook verifies the external snapshot, not a fresh near-duplicate audit of changed reference collections.


In [ ]:
external = pd.read_csv(EXTERNAL_CSV, dtype={"id": str})
audit = pd.read_csv(EXTERNAL_ROOT / "audit/image_audit.csv", dtype={"id": str})
technical = json.loads((EXTERNAL_ROOT / "audit/technical_summary.json").read_text())
removals = json.loads((EXTERNAL_ROOT / "audit/removal_manifest.json").read_text())

require({"id", TARGET, "source", "label_source"} <= set(external.columns), "External schema is incomplete.")
require(external["id"].notna().all() and external["id"].is_unique, "External IDs must be present and unique.")
require(external["id"].str.fullmatch(r"[0-9]+").all(), "External IDs must be numeric filename stems.")
require(external[TARGET].notna().all(), "Missing external articleType labels.")
require(sha256(EXTERNAL_CSV) == technical["labels_sha256"] == removals["labels_sha256"],
        "External labels differ from the cleaned audit snapshot; re-audit before proceeding.")
require(len(external) == removals["remaining_label_rows"] == 1158, "Expected the approved 1,158-row snapshot.")
require(external[TARGET].value_counts().to_dict() == removals["remaining_class_counts"],
        "External class counts differ from cleanup record.")
require(audit["id"].is_unique and set(audit["id"]) == set(external["id"]), "Image audit IDs differ.")
removed_ids = {str(row["id"]) for row in removals["removed"]}
require(not (set(external["id"]) & removed_ids), "A removed image was reintroduced.")
image_paths = {p.stem: p for p in (EXTERNAL_ROOT / "images").glob("*.jpg")}
require(set(image_paths) == set(external["id"]), "Missing or unlabelled external JPEGs.")
audit_by_id = audit.set_index("id")
image_hashes = {}
for image_id in sorted(image_paths):
    path = image_paths[image_id]
    image_hashes[image_id] = sha256(path)
    require(image_hashes[image_id] == audit_by_id.at[image_id, "file_sha256"],
            f"Image changed since audit: {path.name}")
    require(external.set_index("id").at[image_id, TARGET] == audit_by_id.at[image_id, TARGET],
            f"Image audit label differs: {path.name}")
    with Image.open(path) as image:
        transformed = standardize_image(image)
        require(transformed.mode == "RGB" and transformed.size == IMAGE_TARGET_SIZE,
                f"Invalid transformed image: {path.name}")

image_set_hash = hashlib.sha256(
    "\n".join(f"{i}:{image_hashes[i]}" for i in sorted(image_hashes)).encode()
).hexdigest()
require(image_set_hash == technical["image_set_sha256"] == removals["image_set_sha256"],
        "External image collection fingerprint differs from cleanup record.")
print("Verified all", len(external), "retained images and labels.")


## 2. Freeze the baseline split, then append external rows

The original row order, group IDs, split function, seed, and validation share are preserved. External rows receive file-hash group IDs and source-aware repository-relative paths. Unverified external gender/season/usage and other auxiliary attributes are left missing; only articleType is used as supervision. The original external CSV remains untouched.


In [ ]:
base = load_manifest(TARGET)
require(base["id"].is_unique and base["group_id"].is_unique, "EDA manifest IDs/groups must be unique.")
require(not (set(base["id"].astype(str)) & set(external["id"])), "External IDs collide with supplied IDs.")
baseline_train, baseline_val = make_split(
    base, TARGET, validation_share=VALIDATION_SHARE, random_state=SPLIT_SEED
)
require(set(external[TARGET]) <= set(baseline_train[TARGET]), "External data introduces an unknown class.")

def portable(frame):
    result = frame.drop(columns=["path"]).copy()
    result["source"] = "supplied"
    result["label_source"] = "supplied_manifest"
    result["relative_path"] = frame["path"].map(
        lambda p: Path(p).resolve().relative_to(REPO_ROOT).as_posix()
    )
    return result

supplied_train = portable(baseline_train)
validation = portable(baseline_val)
addition = pd.DataFrame(index=range(len(external)), columns=supplied_train.columns)
addition["id"] = external["id"].astype(base["id"].dtype)
addition[TARGET] = external[TARGET].to_numpy()
addition["filename"] = external["id"] + ".jpg"
addition["group_id"] = external["id"].map(image_hashes)
addition["source"] = external["source"].to_numpy()
addition["label_source"] = "external_original_articleType; visual_screened_not_authenticated"
addition["relative_path"] = external["id"].map(
    lambda i: image_paths[i].relative_to(REPO_ROOT).as_posix()
)
require(addition["group_id"].is_unique, "External exact duplicates detected.")
require(not (set(addition["group_id"]) & set(base["group_id"])),
        "External image hashes collide with supplied groups.")
training = pd.concat([supplied_train, addition], ignore_index=True)

require(training["id"].is_unique and training["group_id"].is_unique, "Combined IDs/groups must be unique.")
require(not (set(training["group_id"]) & set(validation["group_id"])), "Training/validation group overlap.")
require(not (set(training["relative_path"]) & set(validation["relative_path"])), "Training/validation path overlap.")
require(validation["source"].eq("supplied").all(), "External row leaked into validation.")
pd.testing.assert_frame_equal(validation, portable(baseline_val))
pd.testing.assert_frame_equal(
    training.iloc[:len(supplied_train)].reset_index(drop=True),
    supplied_train, check_dtype=False,
)
for relative in pd.concat([training["relative_path"], validation["relative_path"]]):
    path = (REPO_ROOT / relative).resolve()
    require(path.is_relative_to(REPO_ROOT) and path.is_file(), f"Missing/unsafe image path: {relative}")

support = pd.concat({
    "supplied_training": baseline_train[TARGET].value_counts(),
    "external_training": addition[TARGET].value_counts(),
    "combined_training": training[TARGET].value_counts(),
    "validation": validation[TARGET].value_counts(),
}, axis=1).fillna(0).astype(int).sort_index()
support.index.name = TARGET
display(support.loc[sorted(external[TARGET].unique())])
print(f"Training: {len(baseline_train):,} + {len(addition):,} = {len(training):,}")
print(f"Validation unchanged: {len(validation):,}")
print("Use supplied_training support for baseline-defined rare-class evaluation buckets.")


## 3. Export portable, versioned manifests

The combined training manifest contains **training rows only**. Do not split it again or replace the global EDA manifest with it. Metadata fingerprints input CSVs, the external image collection, the split settings, and ordered output CSV bytes. Identical reruns reuse identical files; conflicting files are never overwritten. The supplied image collection is assumed to remain the EDA-audited collection.


In [ ]:
def csv_bytes(frame, index=False):
    return frame.to_csv(index=index, lineterminator="\n").encode("utf-8")

payloads = {
    "train_manifest.csv": csv_bytes(training),
    "validation_manifest.csv": csv_bytes(validation),
    "class_support.csv": csv_bytes(support, index=True),
}
metadata = {
    "schema_version": 1,
    "target": TARGET,
    "split_seed": SPLIT_SEED,
    "validation_share": VALIDATION_SHARE,
    "split_policy": "split_supplied_first_then_append_dataset1_to_training_only",
    "base_manifest_sha256": sha256(BASE_CSV),
    "external_labels_sha256": sha256(EXTERNAL_CSV),
    "external_image_set_sha256": image_set_hash,
    "preprocessing_module_sha256": sha256(REPO_ROOT / "src/preprocessing.py"),
    "image_target_size": list(IMAGE_TARGET_SIZE),
    "training_rows": len(training),
    "supplied_training_rows": len(baseline_train),
    "external_training_rows": len(addition),
    "validation_rows": len(validation),
    "classes": sorted(training[TARGET].unique()),
    "path_contract": "relative_path is relative to repository root; resolve into path on each machine",
    "file_sha256": {name: hashlib.sha256(data).hexdigest() for name, data in payloads.items()},
}
version = hashlib.sha256(json.dumps(metadata, sort_keys=True).encode()).hexdigest()[:16]
metadata["version"] = version
OUTPUT_DIR = OUTPUT_ROOT / version
payloads["integration_metadata.json"] = (json.dumps(metadata, indent=2, sort_keys=True) + "\n").encode()

# Preflight every destination before writing any file.
for name, data in payloads.items():
    destination = OUTPUT_DIR / name
    if destination.exists():
        require(destination.read_bytes() == data, f"Conflicting existing output: {destination}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, data in payloads.items():
    destination = OUTPUT_DIR / name
    if not destination.exists():
        with destination.open("xb") as stream:
            stream.write(data)
    require(destination.read_bytes() == data, f"Export verification failed: {destination}")
print("Exported verified manifests:", OUTPUT_DIR.relative_to(REPO_ROOT))
print("Data version:", version)


## 4. Load the exports for a future enriched Task 1 run

This example reconstructs the `path` column expected by the shared image loader. It does not run training or modify existing workers. When adopting these files, replace the original load/split block with this approach, retain the fixed validation manifest, and fit normalization/class weights on the combined training population only. Use the same shared image transform.

All machines must use the same versioned manifests and dataset1 images. Include the data version in cache/checkpoint identity and use a separate enriched-run directory; regenerate workers after changing their source notebook. The independent-job machine arrangement need not change. External data is Task 1 only, and the three enriched classes still have just eight supplied validation images in the current seed-42 split.


In [ ]:
def load_export(name):
    path = OUTPUT_DIR / name
    require(sha256(path) == metadata["file_sha256"][name], f"Export changed: {name}")
    frame = pd.read_csv(path)
    frame["path"] = frame["relative_path"].map(lambda p: str((REPO_ROOT / p).resolve()))
    return frame

train_frame = load_export("train_manifest.csv")
val_frame = load_export("validation_manifest.csv")
require(len(train_frame) == len(training) and len(val_frame) == len(validation), "Reload counts differ.")
require(val_frame["id"].tolist() == baseline_val["id"].tolist(), "Reload changed ordered validation IDs.")
print("Ready for a separate enriched experiment:", len(train_frame), "train /", len(val_frame), "validation")
